In [1]:
import pandas as pd
import yfinance as yfin
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
from scipy import stats

In [2]:
Ticker = ['^NSEI']

# Indicate the start and end dates
start = dt.datetime.now().replace(year=dt.datetime.now().year - 1)
end = dt.datetime.now()

nifty = yfin.download(Ticker, start = start, end = end)
nifty

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2025-06-02,24716.599609,24754.400391,24526.150391,24669.699219,311100
2025-06-03,24542.500000,24845.099609,24502.150391,24786.300781,349300
2025-06-04,24620.199219,24644.250000,24530.449219,24560.449219,280900
2025-06-05,24750.900391,24899.849609,24613.099609,24691.199219,388400
2025-06-06,25003.050781,25029.500000,24671.449219,24748.699219,335600
...,...,...,...,...,...
2026-05-25,24031.699219,24054.449219,23922.849609,23940.250000,351200
2026-05-26,23913.699219,24089.800781,23885.449219,24004.099609,387900


In [3]:
nifty['todayReturns'] = nifty['Close'].pct_change(1) * 100
nifty['tomorrowReturns'] = nifty.todayReturns.shift(-1)

nifty

Price,Close,High,Low,Open,Volume,todayReturns,tomorrowReturns
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,,
Date,,,,,,,
2025-06-02,24716.599609,24754.400391,24526.150391,24669.699219,311100,NaN,-0.704383
2025-06-03,24542.500000,24845.099609,24502.150391,24786.300781,349300,-0.704383,0.316590
2025-06-04,24620.199219,24644.250000,24530.449219,24560.449219,280900,0.316590,0.530870
2025-06-05,24750.900391,24899.849609,24613.099609,24691.199219,388400,0.530870,1.018752
2025-06-06,25003.050781,25029.500000,24671.449219,24748.699219,335600,1.018752,0.400545
...,...,...,...,...,...,...,...
2026-05-25,24031.699219,24054.449219,23922.849609,23940.250000,351200,1.317064,-0.491018
2026-05-26,23913.699219,24089.800781,23885.449219,24004.099609,387900,-0.491018,-0.027385


In [4]:
returns = nifty[['todayReturns', 'tomorrowReturns']].dropna()

returns

Price,todayReturns,tomorrowReturns
Ticker,,
Date,,
2025-06-03,-0.704383,0.316590
2025-06-04,0.316590,0.530870
2025-06-05,0.530870,1.018752
2025-06-06,1.018752,0.400545
2025-06-09,0.400545,0.004186
...,...,...
2026-05-22,0.273102,1.317064
2026-05-25,1.317064,-0.491018


In [5]:
X_no_intercept = returns[['todayReturns']].values
X = np.column_stack([np.ones(len(X_no_intercept)), X_no_intercept]) 
Y = returns[['tomorrowReturns']].values

beta = np.linalg.inv(X.T @ X) @ X.T @ Y

In [6]:
slope, intercept, r_value, p_value, std_err = stats.linregress(returns['todayReturns'], returns['tomorrowReturns'])

In [7]:
print(f"linregress : {slope}")
print(f"numpy : {beta}")

linregress : -0.04656621765051654
numpy : [[-0.01685458]
 [-0.04656622]]


In [8]:
print(f"Slope:     {slope:.6f}")
print(f"Intercept: {intercept:.6f}")
print(f"R²:        {r_value**2:.6f}")
print(f"P-value:   {p_value:.6f}")

Slope:     -0.046566
Intercept: -0.016855
R²:        0.002170
P-value:   0.468839


In [9]:
# Test if Nifty mean daily return significantly different from 0
# Null hypothesis: mean = 0
# Alternate Hypthesis: mean != 0
# level of significance = 0.05
dailyReturns = nifty['todayReturns'].dropna()
dailyReturns

Date
2025-06-03   -0.704383
2025-06-04    0.316590
2025-06-05    0.530870
2025-06-06    1.018752
2025-06-09    0.400545
                ...   
2026-05-25    1.317064
2026-05-26   -0.491018
2026-05-27   -0.027385
2026-05-29   -1.503318
2026-06-01   -0.590290
Name: todayReturns, Length: 245, dtype: float64

In [10]:
print(dailyReturns.mean())
print(dailyReturns.std())

-0.01889323186515401
0.81308129687611


In [11]:
testResults = stats.ttest_1samp(dailyReturns, popmean = 0)
print(testResults)

Ttest_1sampResult(statistic=-0.3637100699428667, pvalue=0.7163894627103031)


In [12]:
#since p-value is > 0.05, null hypothesis cannot be rejected
# mean return is 0